# Qwen2.5-Omni-7B Thinker LoRA 파인튜닝

데비&마를렌 캐릭터 말투 학습

## 준비물
- `train.json` -> Google Drive 루트(내 드라이브 최상단)에 업로드
- HuggingFace 토큰 (https://huggingface.co/settings/tokens)

## 사용법
1. 아래 **설정** 셀에서 HF_TOKEN 입력
2. 런타임 > 모두 실행


In [ ]:
#@title 1. 설정 (이것만 수정)

CHARACTER = "debi-marlene"
HF_TOKEN = ""  #@param {type:"string"}

# 학습 설정
NUM_EPOCHS = 2
LEARNING_RATE = 2e-5
LORA_RANK = 16
LORA_ALPHA = 32
BATCH_SIZE = 2
GRAD_ACCUM = 4
MAX_LENGTH = 512

# 경로
DATA_DIR = "/content/data"
OUTPUT_DIR = f"/content/output/{CHARACTER}"
DRIVE_BACKUP = f"/content/drive/MyDrive/qwen25_omni_backup/{CHARACTER}"

import os
print(f"실질 배치: {BATCH_SIZE * GRAD_ACCUM}")
print(f"학습률: {LEARNING_RATE}")


In [ ]:
#@title 2. GPU 확인
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
assert torch.cuda.is_available()


In [ ]:
#@title 3. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_BACKUP, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
#@title 4. LLaMA-Factory 설치 (5분 소요)
# flash-attn은 빌드 오래 걸려서 제외 (없어도 학습 가능)
!pip install -q "transformers>=4.52.4"
!pip install -q "llamafactory[metrics]>=0.9.2"
!pip install -q qwen-omni-utils[decord]

print("\n=== 설치 완료 ===")
!llamafactory-cli version


In [ ]:
#@title 5. HuggingFace 로그인
if HF_TOKEN:
    !huggingface-cli login --token {HF_TOKEN}
    print("로그인 완료")
else:
    print("[!] HF_TOKEN을 설정 셀에서 입력하세요")


---
## 데이터 준비

train.json을 **Google Drive 루트**(내 드라이브 최상단)에 업로드한 뒤 아래 셀 실행


In [ ]:
#@title 4. train_final.json 가져오기
import shutil, json

search_paths = [
    "/content/drive/MyDrive/train_final.json",
    f"/content/drive/MyDrive/{CHARACTER}/train_final.json",
    f"{DRIVE_BACKUP}/train_final.json",
]

found = False
for p in search_paths:
    if os.path.exists(p):
        shutil.copy2(p, f"{DATA_DIR}/train.json")
        print(f"복사 완료: {p}")
        found = True
        break

if not found and os.path.exists(f"{DATA_DIR}/train.json"):
    print("이미 있음")
    found = True

if not found:
    print("[!] train_final.json을 Google Drive 루트에 업로드하세요")
else:
    with open(f"{DATA_DIR}/train.json") as f:
        data = json.load(f)
    print(f"데이터: {len(data)}개")
    roles = [m["role"] for m in data[0]["messages"]]
    print(f"구조: {roles}")
    print(f"샘플: {json.dumps(data[0], ensure_ascii=False)[:300]}")


In [ ]:
#@title 5. dataset_info.json 생성
import json

dataset_info = {
    CHARACTER: {
        "file_name": "train.json",
        "formatting": "sharegpt",
        "columns": {
            "messages": "messages"
        },
        "tags": {
            "role_tag": "role",
            "content_tag": "content",
            "user_tag": "user",
            "assistant_tag": "assistant",
            "system_tag": "system"
        }
    }
}

with open(f"{DATA_DIR}/dataset_info.json", "w") as f:
    json.dump(dataset_info, f, indent=2)

print("생성 완료")
print(json.dumps(dataset_info, indent=2))


---
## 학습


In [ ]:
#@title 8. 학습 설정 YAML 생성
yaml_config = f"""### Qwen2.5-Omni Thinker LoRA

### 모델
model_name_or_path: Qwen/Qwen2.5-Omni-7B
trust_remote_code: true

### 학습 방법
stage: sft
do_train: true
finetuning_type: lora
lora_rank: {LORA_RANK}
lora_alpha: {LORA_ALPHA}
lora_target: all
freeze_vision_tower: true

### 데이터
dataset_dir: {DATA_DIR}
dataset: {CHARACTER}
template: qwen2_omni
cutoff_len: {MAX_LENGTH}

### 학습 파라미터
output_dir: {OUTPUT_DIR}
overwrite_output_dir: true
per_device_train_batch_size: {BATCH_SIZE}
gradient_accumulation_steps: {GRAD_ACCUM}
learning_rate: {LEARNING_RATE}
num_train_epochs: {NUM_EPOCHS}
lr_scheduler_type: cosine
warmup_ratio: 0.05
bf16: true
logging_steps: 5
save_steps: 100
save_total_limit: 3
gradient_checkpointing: true
report_to: none
"""

with open("/content/train_config.yaml", "w") as f:
    f.write(yaml_config)

print(yaml_config)


In [ ]:
#@title 9. 학습 시작
!llamafactory-cli train /content/train_config.yaml


---
## 테스트


In [ ]:
#@title 10. 추론 테스트
from llamafactory.chat import ChatModel

chat_model = ChatModel({
    "model_name_or_path": "Qwen/Qwen2.5-Omni-7B",
    "adapter_name_or_path": OUTPUT_DIR,
    "template": "qwen2_omni",
    "finetuning_type": "lora",
})

SYSTEM = "너는 이터널 리턴의 쌍둥이 실험체 데비&마를렌이야. 데비(언니)는 활발하고 천진난만하며 장난스러운 말투를 써. 마를렌(동생)은 냉소적이고 신중하며 간결한 말투를 써. 둘이 같이 말할 수도 있고, 데비만 또는 마를렌만 대답할 수도 있어."

questions = ["안녕!", "너 누구야?", "10킬 했어!", "양궁장이네", "계속 져...", "데비야 결혼해줘", "마를렌 좋아해", "심심해"]

for q in questions:
    r = chat_model.chat(
        messages=[{"role": "user", "content": q}],
        system=SYSTEM
    )
    print(f"Q: {q}")
    print(f"A: {r}")
    print()


---
## 저장


In [ ]:
#@title 11. Google Drive 백업
import shutil, glob

backup_target = f"{DRIVE_BACKUP}/adapter"
if os.path.exists(backup_target):
    shutil.rmtree(backup_target)
shutil.copytree(OUTPUT_DIR, backup_target,
    ignore=shutil.ignore_patterns('optimizer*', 'scheduler*', 'global_step*'))
shutil.copy2(f"{DATA_DIR}/train.json", DRIVE_BACKUP)

print(f"백업 완료: {DRIVE_BACKUP}")
for f in sorted(glob.glob(f"{backup_target}/*")):
    size = os.path.getsize(f) / 1024 / 1024
    print(f"  {os.path.basename(f)} ({size:.1f} MB)")


---
## 문제 해결

| 문제 | 해결 |
|------|------|
| OOM (메모리 부족) | MAX_LENGTH=256, BATCH_SIZE=1 |
| 학습 후 성능 저하 | LEARNING_RATE=1e-5, NUM_EPOCHS=1 |
| KeyError: 'qwen2_5_omni' | transformers >= 4.52.4 필요 |
